In [ ]:
# ── Шаг 1: Установка и импорты ────────────────────────────────────
!pip install -q transformers torch tqdm

import random
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from transformers import pipeline
from google.colab import drive
drive.mount('/content/drive')

# ── Шаг 2: Загрузка выборки из оригинальных текстов ───────────────
TXT_DIR = Path('/content/drive/MyDrive/SFU 4/VKR/csvs/TXT_OUTPUT')

# Смотрим что там есть
print('Содержимое TXT_OUTPUT:')
for p in sorted(TXT_DIR.iterdir()):
    print(f'  {p.name} — {len(list(p.glob("*.txt"))) if p.is_dir() else "файл"}')

In [ ]:
# ── Шаг 3: Выборка 50 текстов из каждой папки = 300 итого ─────────
FOLDERS = ['redd_p', 'redd_c', 'reda_p', 'reda_c', 'tgd_p', 'tga_p']
SAMPLE_PER_FOLDER = 50
random.seed(42)

records = []
for folder in FOLDERS:
    folder_path = TXT_DIR / folder
    files = list(folder_path.glob('*.txt'))
    sample = random.sample(files, min(SAMPLE_PER_FOLDER, len(files)))
    print(f'{folder}: {len(files)} файлов → берём {len(sample)}')
    for fpath in sample:
        try:
            text = fpath.read_text(encoding='utf-8').strip()
            if len(text) < 10:
                continue
            records.append({
                'filename': fpath.name,
                'folder': folder,
                'hashtag_label': 'depression' if 'd' in folder else 'anxiety',
                'source': 'reddit' if folder.startswith('red') else 'telegram',
                'type': 'comment' if folder.endswith('_c') else 'post',
                'text': text
            })
        except Exception as e:
            print(f'Ошибка {fpath.name}: {e}')

df = pd.DataFrame(records)
print(f'\nИтого: {len(df)} текстов')
print(df['folder'].value_counts())

# ── Шаг 4: Загружаем mDeBERTa (CPU-friendly) ──────────────────────
print('\nЗагружаем модель...')
classifier = pipeline(
    'zero-shot-classification',
    model='joeddav/xlm-roberta-large-xnli',  # ← та самая которую ты изучала
    device=-1
)
print('Модель загружена ✓')

# ── Шаг 5: Классификация ──────────────────────────────────────────
CANDIDATE_LABELS = [
    'депрессия, грусть, подавленность, безнадёжность, апатия',
    'тревога, страх, беспокойство, паника, тревожность',
    'нейтральная тема, не связанная с психическим здоровьем'
]
LABEL_MAP = {
    CANDIDATE_LABELS[0]: 'depression',
    CANDIDATE_LABELS[1]: 'anxiety',
    CANDIDATE_LABELS[2]: 'neutral'
}

BATCH_SIZE = 8
texts = df['text'].tolist()
all_labels, all_scores = [], []

print(f'Классифицируем {len(texts)} текстов...')
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = [t[:500] for t in texts[i:i+BATCH_SIZE]]
    results = classifier(batch, candidate_labels=CANDIDATE_LABELS, multi_label=False)
    for r in results:
        all_labels.append(LABEL_MAP[r['labels'][0]])
        all_scores.append({LABEL_MAP[l]: s for l, s in zip(r['labels'], r['scores'])})

df['label_zeroshot'] = all_labels
df['zs_score_dep'] = [s['depression'] for s in all_scores]
df['zs_score_anx'] = [s['anxiety']    for s in all_scores]
df['zs_score_neu'] = [s['neutral']    for s in all_scores]
df['zs_confidence'] = df[['zs_score_dep','zs_score_anx','zs_score_neu']].max(axis=1)

# ── Шаг 6: Результаты ─────────────────────────────────────────────
print('\n── Результаты zero-shot ──')
print(df['label_zeroshot'].value_counts())
print(f'Средняя уверенность: {df["zs_confidence"].mean():.3f}')

# Сохраняем
df.to_csv(
    '/content/drive/MyDrive/SFU 4/VKR/zeroshot_300_results.csv',
    index=False, encoding='utf-8-sig'
)
print('Сохранено ✓')